# EBI BioImage Archive — Upload Notebook

**Dataset:** 2026 Multicolour Bayer-SMLM paper  
**Source:** SMB share on intelliflash  
**Destination:** `ftp-private.ebi.ac.uk` (BioImage Archive Webin FTP)

## Workflow
1. **Configure paths** — set SMB root and output directory
2. **Scan + hash** — walk the share, MD5 every TIFF (~hours; run in tmux/screen)
3. **Inspect manifest** — preview `ebi_filelist.tsv` before uploading
4. **Upload** — FTP transfer with size-based resume
5. **Verify** — compare local manifest count to remote

> **Credentials:** read from a local file (default `~/.config/pyS3M/ebi_ftp_credentials.txt`,
> see `_ebi_credentials.py`) — never hard-code or commit them.

---
## 0 — Imports

In [3]:
import ftplib
import getpass
import hashlib
import os
import sys
import time
from datetime import timedelta
from pathlib import Path

try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm not available — progress will be printed every 50 files")

print("Python:", sys.version)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


---
## 1 — Configuration

Edit the two paths below if needed; everything else derives from them.

In [ ]:
# ── Mount the SMB share (idempotent — safe to re-run) ─────────────────────────
# Replaces the old assumption that the share is already mounted via Files/
# Nautilus before this notebook is opened -- mounts it here instead, via the
# same GVFS mechanism, so a fresh session doesn't need that manual step first.
import subprocess

SMB_SERVER = "intelliflash-mgmt-b.ch.private.cam.ac.uk"
SMB_SHARE  = "sycamore_asap_server"
SMB_URI    = f"smb://{SMB_SERVER}/{SMB_SHARE}/"


def ensure_smb_mounted(uri: str = SMB_URI) -> None:
    """Mount *uri* via GVFS (gio) unless it's already mounted."""
    listing = subprocess.run(["gio", "mount", "--list"], capture_output=True, text=True)
    if uri.rstrip("/") in listing.stdout:
        print(f"Already mounted: {uri}")
        return
    print(f"Mounting {uri} ...")
    result = subprocess.run(["gio", "mount", uri], capture_output=True, text=True)
    if result.returncode == 0 or "already mounted" in result.stderr.lower():
        print("Mounted.")
    else:
        print(result.stderr.strip())
        raise RuntimeError(
            f"Could not mount {uri} automatically -- gio may need an interactive "
            f'credential prompt; try running `gio mount "{uri}"` directly in a '
            "terminal first, then re-run this cell."
        )


def gvfs_path(server: str, share: str, subpath: str = "") -> Path:
    """Local filesystem path GVFS exposes an SMB mount at, once mounted."""
    base = Path(f"/run/user/{os.getuid()}/gvfs/smb-share:server={server},share={share}")
    return (base / subpath) if subpath else base


ensure_smb_mounted()

In [ ]:
# ── Source (SMB share) ────────────────────────────────────────────────────────
# Prefer the GVFS mount path from the mount cell above; fall back to the
# legacy /scratch/... mount point some machines use instead (an older-style
# fstab/autofs CIFS mount rather than a per-session GVFS one).
_gvfs_root   = gvfs_path(SMB_SERVER, SMB_SHARE, "2026_Multicolour_Paper/Data")
_legacy_root = Path('/scratch/sycamore-asap/2026_Multicolour_Paper/Data')
SMB_ROOT     = _gvfs_root if _gvfs_root.exists() else _legacy_root

# ── Output directory for manifest files ──────────────────────────────────────
# Same GVFS-first, /scratch/...-fallback resolution as SMB_ROOT above.
_gvfs_out   = gvfs_path(SMB_SERVER, SMB_SHARE, "2026_Multicolour_Paper/EBI_Upload")
_legacy_out = Path("/scratch/sycamore-asap/2026_Multicolour_Paper/EBI_Upload")
OUT_DIR     = (_gvfs_out if _gvfs_out.exists() else _legacy_out).resolve()

FILELIST_TSV = OUT_DIR / "ebi_filelist.tsv"       # manifest for EBI portal
UPLOAD_PAIRS = OUT_DIR / "ebi_upload_paths.txt"   # local→remote path pairs

# ── EBI path prefix (preserved inside the submission) ────────────────────────
EBI_PREFIX = "2026_Multicolour_Paper"

# ── FTP credentials (EBI-issued) ────────────────────────────────────────────
# Read from a local credentials file -- never stored in this git-tracked notebook.
# See _ebi_credentials.py for where the file lives and its format.
from _ebi_credentials import load_ebi_credentials
_creds   = load_ebi_credentials()
FTP_HOST = "ftp-private.ebi.ac.uk"
FTP_USER = _creds["WEBIN_USER"]
FTP_PASS = _creds["WEBIN_PASS"]
FTP_ROOT = _creds["EBI_FTP_DIR"]   # upload root: the submission's private FTP directory

# ── Sanity check ─────────────────────────────────────────────────────────────
if SMB_ROOT.exists():
    print(f"SMB share found: {SMB_ROOT}")
else:
    print(f"WARNING: SMB share not found at {SMB_ROOT}")
    print("Mount the share before running the scan cells.")

print(f"Output directory : {OUT_DIR}")
print(f"FTP host         : {FTP_HOST}")
print(f"FTP user         : {FTP_USER}")
print("FTP root         : (from the credentials file)")

---
## 2 — Scan files and compute MD5 checksums

This will take **several hours** over SMB (I/O-bound).  
Run this cell in a `tmux` session or leave the notebook open.

Output files:
- `ebi_filelist.tsv` — `filename / md5 / size` manifest for EBI portal  
- `ebi_upload_paths.txt` — `local_path \t remote_path` pairs for the upload cell

In [7]:
def md5sum(path: Path, chunk: int = 1 << 20) -> str:
    """Return hex MD5 of a file, reading in 1 MB chunks."""
    h = hashlib.md5()
    with path.open("rb") as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()


def human_size(n_bytes: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} PB"


def scan_and_hash(smb_root: Path, ebi_prefix: str,
                  filelist_out: Path, pathpairs_out: Path):
    """Walk smb_root, hash every TIFF, write manifest and path-pair files."""
    tifs = sorted(
        p for p in smb_root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
    )
    n = len(tifs)
    print(f"Found {n} TIFF files under {smb_root}")
    if n == 0:
        print("Nothing to do — is the SMB share mounted?")
        return

    total_bytes = sum(p.stat().st_size for p in tifs)
    print(f"Total size: {human_size(total_bytes)}")

    t0 = time.time()
    iterator = tqdm(enumerate(tifs, 1), total=n, unit="file") if HAS_TQDM \
               else enumerate(tifs, 1)

    with filelist_out.open("w") as fl, pathpairs_out.open("w") as pp:
        fl.write("filename\tmd5\tsize\n")
        for i, src in iterator:
            rel      = src.relative_to(smb_root)
            ebi_path = f"{ebi_prefix}/{rel}"
            size     = src.stat().st_size
            cksum    = md5sum(src)
            fl.write(f"{ebi_path}\t{cksum}\t{size}\n")
            pp.write(f"{src}\t{ebi_path}\n")

            if not HAS_TQDM and i % 50 == 0:
                elapsed  = time.time() - t0
                rate     = i / elapsed          # files/s
                eta_s    = (n - i) / rate if rate > 0 else 0
                print(f"  {i}/{n}  elapsed {timedelta(seconds=int(elapsed))}"
                      f"  ETA {timedelta(seconds=int(eta_s))}")

    elapsed = time.time() - t0
    print(f"\nDone in {timedelta(seconds=int(elapsed))}.")
    print(f"Manifest : {filelist_out}  ({filelist_out.stat().st_size:,} bytes)")
    print(f"Path pairs: {pathpairs_out}")


# ── Run ───────────────────────────────────────────────────────────────────────
scan_and_hash(SMB_ROOT, EBI_PREFIX, FILELIST_TSV, UPLOAD_PAIRS)

Found 1342 TIFF files under /scratch/sycamore-asap/2026_Multicolour_Paper/Data
Total size: 1.6 TB


  0%|          | 0/1342 [00:00<?, ?file/s]


Done in 0:44:52.
Manifest : /scratch/sycamore-asap/2026_Multicolour_Paper/EBI_Upload/ebi_filelist.tsv  (255,055 bytes)
Path pairs: /scratch/sycamore-asap/2026_Multicolour_Paper/EBI_Upload/ebi_upload_paths.txt


---
## 2b — Add new folders to an existing manifest (incremental)

Use this instead of re-running the full scan above when new data folders have been added to the share after the initial scan/upload. It only hashes files under the listed subfolders (relative to `SMB_ROOT`) and **appends** any not already present to `ebi_filelist.tsv` / `ebi_upload_paths.txt` — the existing 1300+ already-hashed rows are left untouched, so this is safe to re-run and much faster than a full rescan.

After this cell, run the **Upload** section as normal — it will skip everything already on the server and upload only the new files.

In [ ]:
def scan_and_append(smb_root: Path, ebi_prefix: str, subfolders: list,
                     filelist_out: Path, pathpairs_out: Path):
    """Hash TIFFs under specific subfolders of smb_root and append any new
    entries to an existing manifest / path-pairs file, without re-scanning
    or re-hashing files already recorded there."""
    existing_paths = set()
    if filelist_out.exists():
        with filelist_out.open() as f:
            reader = csv.DictReader(f, delimiter="\t")
            existing_paths = {row["filename"] for row in reader}
    else:
        filelist_out.write_text("filename\tmd5\tsize\n")

    tifs = []
    for sub in subfolders:
        root = smb_root / sub
        if not root.exists():
            print(f"WARNING: folder not found, skipping: {root}")
            continue
        tifs.extend(
            p for p in root.rglob("*")
            if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
        )
    tifs.sort()

    new_tifs = [
        p for p in tifs
        if f"{ebi_prefix}/{p.relative_to(smb_root)}" not in existing_paths
    ]
    n = len(new_tifs)
    print(f"Found {len(tifs)} TIFF files under {len(subfolders)} folder(s), {n} new.")
    if n == 0:
        print("Nothing new to add.")
        return

    total_bytes = sum(p.stat().st_size for p in new_tifs)
    print(f"Total new size: {human_size(total_bytes)}")

    t0 = time.time()
    iterator = tqdm(enumerate(new_tifs, 1), total=n, unit="file") if HAS_TQDM \
               else enumerate(new_tifs, 1)

    with filelist_out.open("a") as fl, pathpairs_out.open("a") as pp:
        for i, src in iterator:
            rel      = src.relative_to(smb_root)
            ebi_path = f"{ebi_prefix}/{rel}"
            size     = src.stat().st_size
            cksum    = md5sum(src)
            fl.write(f"{ebi_path}\t{cksum}\t{size}\n")
            pp.write(f"{src}\t{ebi_path}\n")

            if not HAS_TQDM and i % 50 == 0:
                elapsed = time.time() - t0
                rate    = i / elapsed
                eta_s   = (n - i) / rate if rate > 0 else 0
                print(f"  {i}/{n}  elapsed {timedelta(seconds=int(elapsed))}"
                      f"  ETA {timedelta(seconds=int(eta_s))}")

    elapsed = time.time() - t0
    print(f"\nDone in {timedelta(seconds=int(elapsed))}.")
    print(f"Appended {n} new rows to {filelist_out} and {pathpairs_out}")


# ── New folders to add to this submission ──────────────────────────────────
NEW_SCAN_FOLDERS = [
    "ZWO/20260623_MASSIVECELLS",
    "ZWO/20260624_SAureus_NR4A",
]

scan_and_append(SMB_ROOT, EBI_PREFIX, NEW_SCAN_FOLDERS, FILELIST_TSV, UPLOAD_PAIRS)


---
## 3 — Inspect the manifest

Run this after the scan to check file counts, directory breakdown, and total size before uploading.

In [8]:
import csv
from collections import defaultdict

if not FILELIST_TSV.exists():
    print(f"Manifest not found: {FILELIST_TSV}")
    print("Run the scan cell first.")
else:
    rows = []
    with FILELIST_TSV.open() as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            row["size"] = int(row["size"])
            rows.append(row)

    total_files = len(rows)
    total_size  = sum(r["size"] for r in rows)
    print(f"Total files : {total_files:,}")
    print(f"Total size  : {human_size(total_size)}")
    print()

    # Breakdown by top-level sub-directory (camera / dataset)
    by_dir = defaultdict(lambda: {"count": 0, "bytes": 0})
    for r in rows:
        # filename is  EBI_PREFIX/Camera/SubDir/...  — grab Camera/SubDir
        parts = Path(r["filename"]).parts
        key   = "/".join(parts[1:3]) if len(parts) >= 3 else parts[1]
        by_dir[key]["count"] += 1
        by_dir[key]["bytes"] += r["size"]

    print(f"{'Directory':<45}  {'Files':>6}  {'Size':>10}")
    print("-" * 65)
    for key in sorted(by_dir):
        d = by_dir[key]
        print(f"{key:<45}  {d['count']:>6}  {human_size(d['bytes']):>10}")

    print()
    print("First 5 rows:")
    print(f"  {'filename':<60}  {'md5':>32}  {'size':>12}")
    for r in rows[:5]:
        print(f"  {r['filename']:<60}  {r['md5']:>32}  {human_size(r['size']):>12}")

Total files : 1,342
Total size  : 1.6 TB

Directory                                       Files        Size
-----------------------------------------------------------------
ThorLabs/20250305_DyeImaging                       70     29.3 GB
ThorLabs/20250306_MoreDyes                         71     42.3 GB
ThorLabs/20250310_MoreDyeSlides                    29     16.4 GB
Ximea/AlphaSynPAINT                                38    134.3 GB
Ximea/DyeMixtures                                  80     89.1 GB
Ximea/FRET                                         25     20.3 GB
Ximea/HeLaCellSTORM                                25     99.8 GB
Ximea/MassiveCells                                 20     77.5 GB
Ximea/QDots                                       625    329.3 GB
Ximea/SAureus                                      37    136.1 GB
Ximea/Single_Dye_Experiments                      208    304.5 GB
Ximea/TetraspeckBeads                               2    854.7 MB
ZWO/20260402_ATTO647N_ZWO         

---
## 4 — FTP Upload

### Credentials
Read from the credentials file (see the Configuration cell and `_ebi_credentials.py`).

### Resume logic
Files already on the server at the correct size are **skipped**.  
Re-run the upload cell at any time to continue an interrupted transfer.

In [ ]:
# Credentials are read from the credentials file in the Configuration cell above.
# This cell just confirms what will be used.
print(f"FTP host : {FTP_HOST}")
print(f"FTP user : {FTP_USER}")
print("FTP root : (from the credentials file)")
print(f"Password : {'*' * len(FTP_PASS)}")

In [10]:
UPLOAD_CHUNK = 1 << 20  # 1 MB per FTP block


def remote_size(ftp: ftplib.FTP, path: str) -> int:
    """Return remote file size, or -1 if the file does not exist."""
    try:
        return ftp.size(path)
    except ftplib.error_perm:
        return -1


def ensure_remote_dirs(ftp: ftplib.FTP, remote_path: str):
    """Recursively create all intermediate directories on the FTP server."""
    parts = Path(remote_path).parent.parts
    for i in range(1, len(parts) + 1):
        d = str(Path(*parts[:i]))
        try:
            ftp.mkd(d)
        except ftplib.error_perm:
            pass  # already exists — fine


def upload_file(ftp: ftplib.FTP, local: Path, remote: str) -> str:
    """Upload local → remote, skipping if sizes match. Returns 'skip'/'ok'/'error'."""
    local_size = local.stat().st_size
    if remote_size(ftp, remote) == local_size:
        return "skip"
    ensure_remote_dirs(ftp, remote)
    with local.open("rb") as f:
        ftp.storbinary(f"STOR {remote}", f, blocksize=UPLOAD_CHUNK)
    return "ok"


def load_upload_pairs(path: Path) -> list:
    pairs = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            src, dst = line.split("\t", 1)
            pairs.append((Path(src), dst))
    return pairs


def run_upload(ftp_host: str, ftp_user: str, ftp_pass: str,
               ftp_root: str, pairs_file: Path):
    if not pairs_file.exists():
        print(f"Path-pairs file not found: {pairs_file}")
        print("Run the scan cell first.")
        return

    pairs = load_upload_pairs(pairs_file)
    n     = len(pairs)
    print(f"Uploading {n} files to {ftp_host} …")

    ftp = ftplib.FTP(ftp_host, timeout=60)
    ftp.login(ftp_user, ftp_pass)
    ftp.set_pasv(True)
    print("Login OK")

    counts = {"ok": 0, "skip": 0, "error": 0}
    t0     = time.time()

    iterator = tqdm(enumerate(pairs, 1), total=n, unit="file") if HAS_TQDM \
               else enumerate(pairs, 1)

    for i, (local, rel_remote) in iterator:
        remote = f"{ftp_root}/{rel_remote}"
        try:
            status = upload_file(ftp, local, remote)
        except Exception as exc:
            status = "error"
            print(f"  ERROR [{i}/{n}] {local.name}: {exc}", file=sys.stderr)
            # Attempt reconnect
            try:
                ftp.quit()
            except Exception:
                pass
            time.sleep(5)
            ftp = ftplib.FTP(ftp_host, timeout=60)
            ftp.login(ftp_user, ftp_pass)
            ftp.set_pasv(True)

        counts[status] += 1

        if not HAS_TQDM and i % 10 == 0:
            elapsed = time.time() - t0
            rate    = i / elapsed
            eta_s   = (n - i) / rate if rate > 0 else 0
            print(f"  [{i}/{n}]  ok={counts['ok']}  skip={counts['skip']}"
                  f"  err={counts['error']}"
                  f"  ETA {timedelta(seconds=int(eta_s))}")

    try:
        ftp.quit()
    except Exception:
        pass

    elapsed = time.time() - t0
    print(f"\nFinished in {timedelta(seconds=int(elapsed))}.")
    print(f"  Uploaded : {counts['ok']}")
    print(f"  Skipped  : {counts['skip']}")
    print(f"  Errors   : {counts['error']}")


# ── Run upload ────────────────────────────────────────────────────────────────
run_upload(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, UPLOAD_PAIRS)

Uploading 1342 files to ftp-private.ebi.ac.uk …
Login OK


  0%|          | 0/1342 [00:00<?, ?file/s]


Finished in 9:24:27.
  Uploaded : 1342
  Skipped  : 0
  Errors   : 0


---
## 5 — Verify remote file count

Compare the number of lines in the local manifest against the number of files visible on the FTP server.

> This uses a recursive `NLST` walk which can be slow for large trees.  
> The `lftp` one-liner in the comment below is faster if `lftp` is installed.

In [11]:
# ── Count local manifest lines (excluding header) ─────────────────────────────
if FILELIST_TSV.exists():
    with FILELIST_TSV.open() as f:
        local_count = sum(1 for _ in f) - 1  # subtract header
    print(f"Local manifest: {local_count:,} files")
else:
    print("Manifest not found — run the scan cell first.")
    local_count = None


# ── Count remote files via FTP NLST ──────────────────────────────────────────
def count_remote_files(ftp: ftplib.FTP, remote_dir: str) -> int:
    """Recursively count files under remote_dir using NLST."""
    count = 0
    try:
        entries = ftp.nlst(remote_dir)
    except ftplib.error_perm:
        return 0
    for entry in entries:
        # Heuristic: entries with a '.' near the end are files
        if Path(entry).suffix:  # has an extension → likely a file
            count += 1
        else:  # directory — recurse
            count += count_remote_files(ftp, entry)
    return count


if local_count is not None:
    print("Connecting to FTP server for remote count …")
    ftp = ftplib.FTP(FTP_HOST, timeout=60)
    ftp.login(FTP_USER, FTP_PASS)
    ftp.set_pasv(True)

    remote_dir  = f"{FTP_ROOT}/{EBI_PREFIX}"
    remote_count = count_remote_files(ftp, remote_dir)
    ftp.quit()

    print(f"Remote files : {remote_count:,}")
    if remote_count == local_count:
        print("MATCH — all files present on server.")
    else:
        missing = local_count - remote_count
        print(f"MISMATCH — {missing:,} files missing from server. Re-run upload cell.")

Local manifest: 1,342 files
Connecting to FTP server for remote count …
Remote files : 1,342
MATCH — all files present on server.


---
## Alternative: lftp mirror (recommended if lftp is available)

Run the block below in a terminal (not in the notebook) for a more robust upload.  
`lftp` handles resume, retries, and parallelism automatically.

```bash
set -a; source ~/.config/pyS3M/ebi_ftp_credentials.txt; set +a   # WEBIN_USER, WEBIN_PASS, EBI_FTP_DIR
lftp -u "$WEBIN_USER,$WEBIN_PASS" ftp-private.ebi.ac.uk <<EOF
set ftp:passive-mode true
mirror --reverse --verbose --parallel=4 \
  "/run/user/1000/gvfs/smb-share:server=intelliflash-mgmt-b.ch.private.cam.ac.uk,share=sycamore_asap_server/2026_Multicolour_Paper/Data" \
  $EBI_FTP_DIR/2026_Multicolour_Paper
bye
EOF
```

After the mirror completes, count remote files:

```bash
lftp -u "$WEBIN_USER,$WEBIN_PASS" ftp-private.ebi.ac.uk \
  -e "find $EBI_FTP_DIR/2026_Multicolour_Paper | wc -l; bye"
```

Compare against the local manifest:

```bash
# subtract 1 for the header line
echo "Local: $(( $(wc -l < ebi_filelist.tsv) - 1 )) files"
```

---
## 6 — Regenerate the checksums manifest and upload it to the EBI

Run this **after** the upload section above has finished (including any newly added folders) and the remote count matches the local manifest. This rebuilds `biostudies_checksums.json` from the full, current `ebi_filelist.tsv` and re-uploads it to `Checksums/biostudies_checksums.json` on the EBI FTP area, overwriting the previous copy.

In [ ]:
import json

ACCESSION = "S-BIAD3210"


def build_checksums_json(filelist_tsv: Path, out_path: Path, accession: str) -> dict:
    """Build biostudies_checksums.json ({path: {md5, size}}) from the TSV manifest."""
    files = {}
    total_bytes = 0
    with filelist_tsv.open() as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            size = int(row["size"])
            files[row["filename"]] = {"md5": row["md5"], "size": size}
            total_bytes += size

    manifest = {
        "_meta": {
            "description": (
                "MD5 checksums for files deposited under the 2026 Multicolour "
                "Bayer-SMLM BioImage Archive submission."
            ),
            "accession": accession,
            "algorithm": "md5",
            "n_files": len(files),
            "total_bytes": total_bytes,
        },
        "files": files,
    }
    out_path.write_text(json.dumps(manifest, indent=2))
    print(f"Wrote {out_path}  ({len(files):,} files, {human_size(total_bytes)})")
    return manifest


CHECKSUMS_JSON = OUT_DIR / "biostudies_checksums.json"
build_checksums_json(FILELIST_TSV, CHECKSUMS_JSON, ACCESSION)


In [ ]:
def upload_checksums(ftp_host: str, ftp_user: str, ftp_pass: str, ftp_root: str,
                      local_path: Path,
                      remote_rel: str = "2026_Multicolour_Paper/Checksums/biostudies_checksums.json") -> bool:
    """Upload the checksums manifest to the EBI FTP area and verify the size matches."""
    remote = f"{ftp_root}/{remote_rel}"
    ftp = ftplib.FTP(ftp_host, timeout=60)
    ftp.login(ftp_user, ftp_pass)
    ftp.set_pasv(True)
    ensure_remote_dirs(ftp, remote)
    with local_path.open("rb") as f:
        ftp.storbinary(f"STOR {remote}", f, blocksize=UPLOAD_CHUNK)
    ok = remote_size(ftp, remote) == local_path.stat().st_size
    ftp.quit()
    print(f"Remote path: {remote}")
    print("MATCH — checksums manifest uploaded correctly." if ok
          else "MISMATCH — re-run this cell.")
    return ok


upload_checksums(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, CHECKSUMS_JSON)


---
## 7 — Regenerate the EBI file-list JSON (for the portal)

Run this last, after the checksums manifest above has been rebuilt. It writes `2026_Multicolour_Filelist.json` to `OUT_DIR` (the `EBI_Upload/` folder on the SMB share) in the BioStudies file-listing schema, covering every file in the current `ebi_filelist.tsv` plus the checksums manifest itself. This file is **not** uploaded automatically — submit it through the BioImage Archive portal by hand.

In [ ]:
def build_ebi_filelist_json(filelist_tsv: Path, checksums_path: Path,
                             ebi_prefix: str, out_path: Path) -> list:
    """Build the flat BioStudies file-listing JSON from the TSV manifest,
    plus one entry for the checksums manifest itself."""
    entries = []
    with filelist_tsv.open() as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            size = int(row["size"])
            entries.append({
                "path": row["filename"],
                "size": size,
                "attributes": [{"name": "size", "value": str(size)}],
                "type": "file",
            })

    cksum_size = checksums_path.stat().st_size
    entries.append({
        "path": f"{ebi_prefix}/Checksums/{checksums_path.name}",
        "size": cksum_size,
        "attributes": [{"name": "size", "value": str(cksum_size)}],
        "type": "file",
    })

    out_path.write_text(json.dumps(entries, indent=2))
    print(f"Wrote {out_path}  ({len(entries):,} entries)")
    return entries


FILELIST_JSON = OUT_DIR / "2026_Multicolour_Filelist.json"
build_ebi_filelist_json(FILELIST_TSV, CHECKSUMS_JSON, EBI_PREFIX, FILELIST_JSON)


---
## Checklist

- [ ] EBI Webin account created and credentials to hand
- [ ] SMB share mounted at the path in cell 1
- [ ] Scan cell run → `ebi_filelist.tsv` + `ebi_upload_paths.txt` produced
- [ ] Manifest inspected (file count and size look right)
- [ ] MD5s spot-checked manually on a few files: `md5sum <file>`
- [ ] Upload complete (errors = 0)
- [ ] Remote count matches manifest count
- [ ] New folders added via §2b (`ZWO/20260623_MASSIVECELLS`, `ZWO/20260624_SAureus_NR4A`) scanned and uploaded
- [ ] Checksums manifest rebuilt and re-uploaded (§6)
- [ ] `2026_Multicolour_Filelist.json` regenerated (§7)
- [ ] `ebi_filelist.tsv` submitted in the BioImage Archive portal
- [ ] Metadata fields completed in portal
- [ ] Accession number recorded in `claude/EBI_submission.md`: **S-BIADXXXXXXX**